# Stacking Ensemble

In [ ]:
# import numpy as np
# import pandas as pd

# lstm_model_train = pd.read_csv("lstm_model_train.csv")
# logistic_model_train = pd.read_csv("logistic_model_train.csv")
# mlp_model_train = pd.read_csv("mlp_model_train.csv")
# random_forest_model_train = pd.read_csv("random_forest_model_train.csv")
# naive_bayes_model_train = pd.read_csv("naive_bayes_model_train.csv")
# cnn_model_train = pd.read_csv("cnn_model_train.csv")
# svm_model_train = pd.read_csv("svm_model_train.csv")

# #overall training df
# X_train = pd.concat([lstm_model_train, logistic_model_train, mlp_model_train, random_forest_model_train, naive_bayes_model_train, cnn_model_train, svm_model_train], axis=1)
# y_train = X_train["label"]
# X_train = X_train.drop(["label"], axis=1)



# lstm_model_valid = pd.read_csv("lstm_model_valid.csv")
# logistic_model_valid = pd.read_csv("logistic_model_valid.csv")
# mlp_model_valid = pd.read_csv("mlp_model_valid.csv")
# random_forest_model_valid = pd.read_csv("random_forest_model_valid.csv")
# naive_bayes_model_valid = pd.read_csv("naive_bayes_model_valid.csv")
# cnn_model_valid = pd.read_csv("cnn_model_valid.csv")
# svm_model_valid = pd.read_csv("svm_model_valid.csv")

# #overall validation df
# X_valid = pd.concat([lstm_model_valid, logistic_model_valid, mlp_model_valid, random_forest_model_valid, naive_bayes_model_valid, cnn_model_valid, svm_model_valid], axis=1)
# y_valid = X_valid["label"]
# X_valid = X_valid.drop(["label"], axis=1)

# lstm_model_test = pd.read_csv("lstm_model_test.csv")
# logistic_model_test = pd.read_csv("logistic_model_test.csv")
# mlp_model_test = pd.read_csv("mlp_model_test.csv")
# random_forest_model_test = pd.read_csv("random_forest_model_test.csv")
# naive_bayes_model_test = pd.read_csv("naive_bayes_model_test.csv")
# cnn_model_test = pd.read_csv("cnn_model_test.csv")
# svm_model_test = pd.read_csv("svm_model_test.csv")

# #overall test df
# X_test = pd.concat([lstm_model_test, logistic_model_test, mlp_model_test, random_forest_model_test, naive_bayes_model_test, cnn_model_test, svm_model_test], axis=1)


# #NOTE: assume csvs have a column named pred which has their prediction for the sample

In [ ]:
# IMPORT OUR MODELS
lstm_model = ...
logistic_model = ...
mlp_model = ...
random_forest_model = ...
naive_bayes_model = ...
cnn_model = ...
svm_model = ...

## general stacking

In [ ]:
from mlxtend.classifier import StackingClassifier
from sklearn.linear_model import LogisticRegression

#logic:
#set 1: train base learners
#set 2: train meta learner on base learner predictions
#set 3: test meta learner
meta = LogisticRegression(max_iter=500)



stack = StackingClassifier(
    classifiers=[
        lstm_model,
        logistic_model,
        mlp_model,
        random_forest_model,
        naive_bayes_model,
        cnn_model,
        svm_model
    ],
    meta_classifier=meta,
    use_probas=True,
    cv=5
)

stack.fit(X_valid, y_valid)

# valid_pred = stack.predict(X_valid)
test_pred  = stack.predict(X_test)

print("Validation:", accuracy_score(y_valid, valid_pred))
print("Test:",       accuracy_score(y_test, test_pred))


## find best hyperparameters

In [ ]:
from sklearn.metrics import accuracy_score

penalties = ["l1", "l2", "elasticnet"]
Cs = [0.01, 0.1, 1.0, 10]
l1_ratios = [0.2, 0.5, 0.8]    

results = []

best_acc = 0
best_params = None
best_stack = None

for penalty in penalties:
    for C in Cs:
        if penalty == "elasticnet":
            for l1 in l1_ratios:
                meta = LogisticRegression(
                    max_iter=1000,
                    penalty="elasticnet",
                    solver="saga",
                    l1_ratio=l1,
                    C=C
                )

                stack = StackingClassifier(
                    classifiers=[
                        lstm_model,
                        logistic_model,
                        mlp_model,
                        random_forest_model,
                        naive_bayes_model,
                        cnn_model,
                        svm_model
                    ],
                    meta_classifier=meta,
                    use_probas=True,
                    cv=5
                )

                stack.fit(X_train, y_train)
                val_pred = stack.predict(X_valid)
                acc = accuracy_score(y_valid, val_pred)

                results.append((penalty, C, l1, acc))
                print(f"penalty={penalty}, C={C}, l1={l1}, val_acc={acc:.4f}")

                if acc > best_acc:
                    best_acc = acc
                    best_params = (penalty, C, l1)
                    best_stack = stack

        # L2 (simpler case)
        else:
            meta = LogisticRegression(
                max_iter=1000,
                penalty="l2",
                solver="lbfgs",
                C=C
            )

            stack = StackingClassifier(
                classifiers=[
                    lstm_model,
                    logistic_model,
                    mlp_model,
                    random_forest_model,
                    naive_bayes_model,
                    cnn_model,
                    svm_model
                ],
                meta_classifier=meta,
                use_probas=True,
                cv=5
            )

            stack.fit(X_train, y_train)
            val_pred = stack.predict(X_valid)
            acc = accuracy_score(y_valid, val_pred)

            results.append((penalty, C, None, acc))
            print(f"penalty={penalty}, C={C}, val_acc={acc:.4f}")

            if acc > best_acc:
                best_acc = acc
                best_params = (penalty, C, None)
                best_stack = stack


print("\nBest params:", best_params)
print("Best validation accuracy:", best_acc)

test_pred = best_stack.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, test_pred))


## probably unnecessary

In [ ]:
meta = LogisticRegression()
meta.fit(base_valid.values, y_valid)
test_pred = meta.predict(X_test.values)
print("Validation accuracy:", accuracy_score(y_valid, val_pred))
test_pred = meta.predict(base_test.values)
print("Test accuracy:", accuracy_score(y_test, test_pred))
